# Phase 4a: Cross-layer SAE matching

Two pretrained SAEs from `gpt2-small-res-jb`, layers 7 and 8 of GPT-2-small.
Both are 24,576 features over a 768-dim residual stream.

We compare four methods:
1. Cosine-Hungarian on decoder directions.
2. Activation-Hungarian on per-feature firing patterns.
3. Entropic GW on within-SAE cosine distance.
4. Entropic Fused-GW (combines cross-side cosine + within-side cosine), swept over alpha.

Evaluation: matched-pair activation correlation on held-out tokens.

In [ ]:
from __future__ import annotations

import time

from sae_lens import SAE, HookedSAETransformer

from ot_sae.activations import collect_sae_features, top_active_features
from ot_sae.evaluation import evaluate_matching
from ot_sae.matching import (
    activation_hungarian,
    cosine_hungarian,
    fused_gw_matching,
    gw_matching,
)

device = "cpu"
print(f"device: {device}")

model = HookedSAETransformer.from_pretrained("gpt2", device=device)

sae_7 = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    device=device,
)
sae_8 = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.8.hook_resid_pre",
    device=device,
)

HOOK_7 = "blocks.7.hook_resid_pre"
HOOK_8 = "blocks.8.hook_resid_pre"

print(f"Layer 7 SAE: d_in={sae_7.cfg.d_in}, d_sae={sae_7.cfg.d_sae}")
print(f"Layer 8 SAE: d_in={sae_8.cfg.d_in}, d_sae={sae_8.cfg.d_sae}")

In [ ]:
print("Collecting layer 7 activations...")
fc_7 = collect_sae_features(model, sae_7, HOOK_7, n_docs=100, max_tokens_per_doc=128)
print(f"Layer 7: {fc_7.features.shape}")

print("\nCollecting layer 8 activations...")
fc_8 = collect_sae_features(model, sae_8, HOOK_8, n_docs=100, max_tokens_per_doc=128)
print(f"Layer 8: {fc_8.features.shape}")

In [ ]:
TOP_N = 2000

idx_7 = top_active_features(fc_7, top_k=TOP_N)
idx_8 = top_active_features(fc_8, top_k=TOP_N)

print(f"Top-{TOP_N} features at layer 7: indices in [{idx_7.min()}, {idx_7.max()}]")
print(f"Top-{TOP_N} features at layer 8: indices in [{idx_8.min()}, {idx_8.max()}]")
print(
    f"Overlap of indices (sanity, expect small): "
    f"{len(set(idx_7.tolist()) & set(idx_8.tolist()))}"
)

In [ ]:
W_dec_7 = sae_7.W_dec[idx_7].cpu()
W_dec_8 = sae_8.W_dec[idx_8].cpu()

F_7 = fc_7.features[:, idx_7]
F_8 = fc_8.features[:, idx_8]

print(f"W_dec_7 shape: {W_dec_7.shape}")
print(f"W_dec_8 shape: {W_dec_8.shape}")
print(f"F_7 shape: {F_7.shape}")
print(f"F_8 shape: {F_8.shape}")

In [ ]:
n_tokens = F_7.shape[0]
n_train = int(0.7 * n_tokens)

F_7_train = F_7[:n_train]
F_8_train = F_8[:n_train]
F_7_eval = F_7[n_train:]
F_8_eval = F_8[n_train:]

print(f"Train tokens: {n_train}, Eval tokens: {n_tokens - n_train}")

In [ ]:
print("Computing cosine-Hungarian matching...")
t0 = time.time()
m_cosine = cosine_hungarian(W_dec_7, W_dec_8)
print(f"  done in {time.time() - t0:.1f}s")

print("Computing activation-Hungarian matching...")
t0 = time.time()
m_activation = activation_hungarian(F_7_train, F_8_train)
print(f"  done in {time.time() - t0:.1f}s")

print("Computing entropic GW matching...")
t0 = time.time()
m_gw = gw_matching(W_dec_7, W_dec_8, epsilon=5e-3, max_iter=500)
print(f"  done in {time.time() - t0:.1f}s")

In [ ]:
print("Cosine-Hungarian:")
res_c = evaluate_matching(m_cosine, F_7_eval, F_8_eval)
for k, v in res_c.items():
    print(f"  {k}: {v}")

print("\nActivation-Hungarian:")
res_a = evaluate_matching(m_activation, F_7_eval, F_8_eval)
for k, v in res_a.items():
    print(f"  {k}: {v}")

print("\nEntropic GW:")
res_g = evaluate_matching(m_gw, F_7_eval, F_8_eval)
for k, v in res_g.items():
    print(f"  {k}: {v}")

In [ ]:
# If you patched matching.py while the kernel was running, force a reimport.
import importlib

import ot_sae.matching

importlib.reload(ot_sae.matching)

alpha_values = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
fgw_results: dict[float, dict] = {}

for alpha in alpha_values:
    print(f"\n--- alpha = {alpha} ---")
    t0 = time.time()
    m_fgw = fused_gw_matching(
        W_dec_7,
        W_dec_8,
        alpha=alpha,
        epsilon=5e-3,
        max_iter=500,
        tol=1e-7,
    )
    elapsed = time.time() - t0

    res = evaluate_matching(m_fgw, F_7_eval, F_8_eval)
    res["time_s"] = elapsed
    res["agreement_with_cosine_hungarian"] = float((m_fgw == m_cosine).mean())
    res["agreement_with_activation_hungarian"] = float((m_fgw == m_activation).mean())
    fgw_results[alpha] = res

    print(f"  time:                    {elapsed:.1f}s")
    print(f"  mean_corr:               {res['mean_corr']:.4f}")
    print(f"  median_corr:             {res['median_corr']:.4f}")
    print(f"  frac > 0.5:              {res['frac_above_0p5']:.4f}")
    print(f"  agreement w/ cos-Hung:   {res['agreement_with_cosine_hungarian']:.4f}")
    print(f"  agreement w/ act-Hung:   {res['agreement_with_activation_hungarian']:.4f}")

In [ ]:
print("=" * 80)
print(f"{'alpha':>7s}  {'mean_corr':>10s}  {'frac>0.5':>10s}  {'agree-cosH':>12s}  {'time':>8s}")
print("-" * 80)
for a, r in fgw_results.items():
    print(
        f"{a:7.2f}  {r['mean_corr']:10.4f}  {r['frac_above_0p5']:10.4f}  "
        f"{r['agreement_with_cosine_hungarian']:12.4f}  {r['time_s']:7.1f}s"
    )

print("\nBaselines (from cell 8):")
print(
    f"  Cosine-Hungarian       mean_corr={res_c['mean_corr']:.4f}  frac>0.5={res_c['frac_above_0p5']:.4f}"
)
print(
    f"  Activation-Hungarian   mean_corr={res_a['mean_corr']:.4f}  frac>0.5={res_a['frac_above_0p5']:.4f}"
)
print(
    f"  Pure GW (alpha=0)      mean_corr={res_g['mean_corr']:.4f}  frac>0.5={res_g['frac_above_0p5']:.4f}"
)

In [ ]:
import json
from pathlib import Path

OUT_DIR = Path("../results")
OUT_DIR.mkdir(exist_ok=True)

payload = {
    "config": {
        "n_docs": 100,
        "max_tokens_per_doc": 128,
        "top_n": TOP_N,
        "n_train": n_train,
        "n_eval": n_tokens - n_train,
        "epsilon": 5e-3,
        "alpha_values": alpha_values,
    },
    "baselines": {
        "cosine_hungarian": res_c,
        "activation_hungarian": res_a,
        "pure_gw": res_g,
    },
    "fused_gw": {str(a): r for a, r in fgw_results.items()},
}

out_path = OUT_DIR / "phase4a_results.json"
out_path.write_text(json.dumps(payload, indent=2, default=str))
print(f"saved {out_path}")